In [2]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install prophet

Defaulting to user installation because normal site-packages is not writeable
  Using cached prophet-1.4.0-py3-none-win_amd64.whl.metadata (3.7 kB)
  Using cached cmdstanpy-1.3.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached holidays-0.105-py3-none-any.whl.metadata (55 kB)
  Using cached stanio-0.5.1-py3-none-any.whl.metadata (1.6 kB)
Using cached prophet-1.4.0-py3-none-win_amd64.whl (12.1 MB)
Using cached cmdstanpy-1.3.0-py3-none-any.whl (99 kB)
Using cached holidays-0.105-py3-none-any.whl (1.6 MB)
Using cached stanio-0.5.1-py3-none-any.whl (8.1 kB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

def run_forecasting_pipeline(input_path: str):
    print("[+] Initiating Module 3: Time-Series Forecasting Engines...")
    
    # 1. Load Feature-Engineered Dataset
    df = pd.read_csv(input_path, parse_dates=['Date'], low_memory=False)
    df = df.sort_values(by=['Store', 'Date']).reset_index(drop=True)
    
    # 2. Enforce Chronological Train-Test Split (Final 6 Weeks for Holdout Validation)
    max_date = df['Date'].max()
    split_date = max_date - pd.Timedelta(weeks=6)
    
    train_df = df[df['Date'] <= split_date].copy()
    test_df = df[df['Date'] > split_date].copy()
    
    print(f"[+] Validation Cutoff Date: {split_date.strftime('%Y-%m-%d')}")
    print(f"[+] Training Set Records: {len(train_df):,} | Validation Set Records: {len(test_df):,}")
    
    # -------------------------------------------------------------
    # LEVEL 1: Heuristic Baseline (7-Day Moving Average)
    # -------------------------------------------------------------
    test_df['Pred_Moving_Avg'] = test_df['Sales_Rolling_Mean_7']
    ma_mape = mean_absolute_percentage_error(test_df['Sales'], test_df['Pred_Moving_Avg']) * 100
    ma_rmse = np.sqrt(mean_squared_error(test_df['Sales'], test_df['Pred_Moving_Avg']))
    print(f"\n[ Level 1 - Moving Average ] Validation MAPE: {ma_mape:.2f}% | RMSE: {ma_rmse:.2f}")
    
    # -------------------------------------------------------------
    # LEVEL 2: Statistical Baseline (Prophet on Representative Store)
    # -------------------------------------------------------------
    valid_store_id = None
    for store_id in df['Store'].unique():
        s_train = train_df[train_df['Store'] == store_id]
        s_test = test_df[test_df['Store'] == store_id]
        if len(s_train) >= 30 and len(s_test) >= 5:
            valid_store_id = store_id
            break
            
    if valid_store_id is not None:
        p_train = train_df[train_df['Store'] == valid_store_id][['Date', 'Sales']].rename(columns={'Date': 'ds', 'Sales': 'y'}).dropna()
        p_test = test_df[test_df['Store'] == valid_store_id][['Date', 'Sales']].rename(columns={'Date': 'ds', 'Sales': 'y'}).dropna()
        
        m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        m.fit(p_train)
        forecast = m.predict(p_test[['ds']])
        
        prophet_mape = mean_absolute_percentage_error(p_test['y'], forecast['yhat']) * 100
        prophet_rmse = np.sqrt(mean_squared_error(p_test['y'], forecast['yhat']))
        print(f"[ Level 2 - Prophet (Store {valid_store_id}) ] Validation MAPE: {prophet_mape:.2f}% | RMSE: {prophet_rmse:.2f}")
    
    # -------------------------------------------------------------
    # LEVEL 4: Target Model (XGBoost Regressor across ALL Stores)
    # -------------------------------------------------------------
    drop_cols = ['Date', 'Sales', 'Customers', 'StateHoliday', 'StoreType', 'Assortment', 'PromoInterval']
    feature_cols = [c for c in df.columns if c not in drop_cols and np.issubdtype(df[c].dtype, np.number)]
    
    X_train, y_train = train_df[feature_cols], train_df['Sales']
    X_test, y_test = test_df[feature_cols], test_df['Sales']
    
    xgb_model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    print("\n[+] Training XGBoost Regressor across full feature matrix...")
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
    )
    
    test_df['Forecast_Sales'] = xgb_model.predict(X_test)
    
    xgb_mape = mean_absolute_percentage_error(y_test, test_df['Forecast_Sales']) * 100
    xgb_rmse = np.sqrt(mean_squared_error(y_test, test_df['Forecast_Sales']))
    print(f"\n[ Level 4 - XGBoost Engine ] Empirical Validation MAPE: {xgb_mape:.2f}% | RMSE: {xgb_rmse:.2f}")
    
    # Save predictions file for Module 4
    test_df.to_csv("rossmann_forecasts.csv", index=False)
    print(f"\n[+] Forecast predictions saved to 'rossmann_forecasts.csv'. Total Rows: {len(test_df):,}")
    
    return test_df

# Run Module 3 Pipeline
df_forecasts = run_forecasting_pipeline("rossmann_engineered.csv")

[+] Initiating Module 3: Time-Series Forecasting Engines...


Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
02:38:10 - cmdstanpy - INFO - Chain [1] start processing
02:38:10 - cmdstanpy - INFO - Chain [1] done processing


[+] Validation Cutoff Date: 2015-06-19
[+] Training Set Records: 192,088 | Validation Set Records: 20,582

[ Level 1 - Moving Average ] Validation MAPE: 25.38% | RMSE: 2192.79
[ Level 2 - Prophet (Store 2) ] Validation MAPE: 19.55% | RMSE: 1252.55

[+] Training XGBoost Regressor across full feature matrix...
[0]	validation_0-rmse:2637.79248
[100]	validation_0-rmse:1076.98584
[200]	validation_0-rmse:1037.69131
[300]	validation_0-rmse:1017.97526
[400]	validation_0-rmse:1005.54215
[499]	validation_0-rmse:998.14966

[ Level 4 - XGBoost Engine ] Empirical Validation MAPE: 10.92% | RMSE: 998.15

[+] Forecast predictions saved to 'rossmann_forecasts.csv'. Total Rows: 20,582
